<a href="https://colab.research.google.com/github/nikolaicarrion/2026_Intro_Python/blob/main/FEEBACK_NIKOLAI_CARRION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.4-matplotlib-and-xarray-practice-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# Feedback Nikolai Carrion

One of the most challenging exercise was exercise 14 of the chapter 1.4 about plots and xarrays. I will gradually break down the problem encoutered and show how I was able to solve this exercise.

# General Comment

Xarray is a completly new concept that was introduced during this course and was not only hard to visualise what was a DataArray, but it was particularly hard to manipulate the data. During this exercice, eventhough it's broken in several parts, I still faced challenges, especially in the conceptual parts.

Though the synthaxe, most of the time, was somewhat not the main issue, the selection and the manipulation of the data inside the initial Xarray Dataset was really challenging, which later caused logical errors.

In this feedback, I will breakdown the errors encoutered and the thought process to solve them.

## Exercise 14: Select, reduce, and subtract

Build the small Dataset below, then:

1. Print the spatial mean of the first day (by position) and the time mean at each longitude
   along 47.0° N (by label).
2. Select the grid point nearest 46.8° N, 6.3° E, and print its coordinates and its ten values.
3. Subtract each day's spatial mean — the mean over `lat` and `lon` — from the field. Print the
   dims of the result, and its spatial mean on the first day, which should be zero up to rounding.

```python
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})
```

# Part 1

The first step was to study the given code, which contains the Xarray Dataset and the variables which were its building blocks.

The first challenge was to read and understand the Dataset which lead to my first error : using ds["time"] instead of ds["t2m"]. The fact that I didn't get an error was missleading, though the answer was obviously incorrect, since it gave a date instead of a temperature.

Let's break it down:

## Reading and understanding a Dataset

{"t2m": (("time", "lat", "lon"), t)

This line creates a new variable "t2m" that associates our temperature values "t" with the dimensions "time", "lat" and "lon".

To understand what was a dimension in Xarray, this diagram was really helpful : https://docs.xarray.dev/en/stable/_images/dataset-diagram.png


In [1]:
# Provided data

import numpy as np
import xarray as xr

rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})

In [2]:
# Part 1
spatial_mean_day1 = ds["time"].isel(time = 0).mean().item()
print(f"The spatial mean for day 1 is {spatial_mean_day1} °C")

The spatial mean for day 1 is 2024-01-01 00:00:00 °C


This first line of code prints a date. By doing, ds["time"], I'm selecting the time dimension, thinking that each time is directly associated with a temperature, almost as a dictionary.

To correct this line, I simply had to change "time" with "t2m", which was defined ealier in the xr.Dataset.

As mentionned before, it was difficult to understand how a Dataset is built.
"t2m":(("time","lat","lon"), t) mean that you are creating a new variable "t2m", where you then assigne time, lat and lon for every temperature in t. The actual time, lat and lon is then assigned in the line below with coords.

In [3]:
# Day 1
spatial_mean_day1 = ds["t2m"].isel(time =0).mean().item()
print(f"The spatial mean for day 1 is {round(spatial_mean_day1,2)} °C")


# Latitude 47 °N
spatial_mean_day1_lon = ds["t2m"].sel(lat = 47.0).mean().item()
print(f"The spatial mean for lat 47 °N {round(spatial_mean_day1_lon,2)} °C")

The spatial mean for day 1 is 5.07 °C
The spatial mean for lat 47 °N 5.22 °C


# Part 2 - Xarray and data manipulation

Selecting data in an Dataset was also challenging due to several reasons:

- The understanding of dimensions weren't understood very well
- I thought that .sel could include conditions, when selecting multiple dimensions.

Eventhough this was more a synthax issue, the examples in the documentation was handy to solve this selection problem : https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html

In [4]:
# Part 2
point = ds["t2m"].sel((lat == 46.8) & (lon == 6.3) , method= "nearest")
print("Latitude:", point["lat"].item())
print("Longitude:", point["lon"].item())
print("Values:", point.values)

ValueError: operands could not be broadcast together with shapes (3,) (4,) 

## Correction of part 2

We can simply add .sel(lat=x, lon = y) to select a 2D point in our dataset.

In [ ]:
# Correction part 2

point = ds["t2m"].sel(lat = 46.8, lon = 6.3, method= "nearest")

print("Latitude:", point["lat"].item())
print("Longitude:", point["lon"].item())
print("Values:", point.values)

# Part 3 - Spatial mean

Previously, we've shown, via a graph, that a variable can be represented in 3D with the dimensions time, lat and lon. When we selected lat and lon in part 2, there was a remaining set of temperature values along the t axis.

I initially thought that .mean(dim=("time")) gave the spacial mean for each day and reduced the lat and lon axis, which is wrong and it's actually the opposite.  

By doing .mean(dim=("lat", "lon")), we average the 2D space (lat and lon) to a one value temperature per day. This means that we end up with a 1D array, which is supported by the documentation https://xarray.pydata.org/en/v0.8.2/generated/xarray.DataArray.mean.html that explains after using this function, we end up with dimensions removed.

The second error is result.isel(time=0).mean(dim="time").values :
We selected only one value in time which removes this axis, therefore we can't average over this dimensions since it doesn't exist.

In [24]:
# Part 3
result = ds["t2m"] - ds["t2m"].mean(dim=("time"))

print(result.dims)
print(result.isel(time=0).mean(dim="time").values)

('time', 'lat', 'lon')


ValueError: 'time' not found in array dimensions ('lat', 'lon')

## Part 3 - Correction

After changing the dimensions to lat and lon in the .mean() function, we are able to select the first day in the time axis, and we then get 0.

In [28]:
# Part 3
result = ds["t2m"] - ds["t2m"].mean(dim=("lat", "lon"))

print(result.dims)
print(result.isel(time=0).mean(dim=("lat", "lon")).values.round(0))

('time', 'lat', 'lon')
-0.0
